# 09. Two-Tower Retrieval: ALS vs Neural

ALS는 user×item interaction만 사용하지만, Two-Tower는 **사이드 피처**(language, stars 등)를 임베딩에 직접 태울 수 있습니다.

| | ALS | Two-Tower |
|---|---|---|
| 입력 | interaction만 | + repo metadata |
| 학습 | 행렬분해 | in-batch contrastive |
| 라이브러리 | implicit | PyTorch |
| 디바이스 | CPU | CPU |

> **Note:** 이 노트북은 메모리 제약으로 `scripts/train_two_tower.py`로 실행합니다.
> ```bash
> OMP_NUM_THREADS=1 uv run python scripts/train_two_tower.py
> ```

## 실행 결과 (5% sample, 2주 train, 13분)

### Training

```
Epoch 1: loss=6.9311, 68.0s
Epoch 2: loss=4.9313, 85.8s
Epoch 3: loss=1.9624, 59.9s
Epoch 4: loss=0.2880, 62.5s
Epoch 5: loss=0.2024, 75.0s
```

### Retrieval 평가 (5,000 eval users)

| Model | K=10 Precision | K=10 Recall | K=10 NDCG | K=50 NDCG | K=100 NDCG |
|---|---|---|---|---|---|
| Popularity | 0.00000 | 0.00000 | 0.00000 | 0.00001 | 0.00027 |
| ALS | 0.00000 | 0.00000 | 0.00000 | 0.00013 | 0.00016 |
| **Two-Tower** | **0.00004** | **0.00030** | **0.00025** | **0.00035** | **0.00035** |

**Two-Tower가 ALS 대비 NDCG@50에서 2.7배 개선.**

5% 샘플 + 2주 데이터라 절대 수치는 작지만, 사이드 피처(language, stars, forks)를 활용한 neural retrieval이 sparse 데이터에서 collaborative filtering보다 효과적임을 확인.

## 1. 데이터 준비

In [ ]:
%%time
import os
import pickle
import sqlite3
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import sparse
from torch.utils.data import DataLoader, TensorDataset

from gharchive.loader import load_period
from ghrec.recommend import popularity_scores

OUTPUT_DIR = Path("../../data/daily_agg")
MODEL_DIR = Path("../../data/models")
DB_PATH = Path("../../data/repo_metadata.db")

# 2주 train + 1주 test, 5% 샘플
TRAIN_START, TRAIN_END = date(2026, 3, 15), date(2026, 3, 28)
TEST_START, TEST_END = date(2026, 3, 29), date(2026, 4, 3)

WEIGHTS = {
    "WatchEvent": 1.0, "ForkEvent": 2.0, "IssuesEvent": 0.5,
    "PullRequestEvent": 3.0, "IssueCommentEvent": 0.3, "PushEvent": 0.2,
}

SAMPLE_RATIO = 0.05

train_df = load_period(OUTPUT_DIR, TRAIN_START, TRAIN_END)
test_df = load_period(OUTPUT_DIR, TEST_START, TEST_END)

rng = np.random.default_rng(42)
all_users = set(train_df["actor_id"].unique()) | set(test_df["actor_id"].unique())
sampled = set(rng.choice(list(all_users), size=int(len(all_users) * SAMPLE_RATIO), replace=False))
train_df = train_df[train_df["actor_id"].isin(sampled)]
test_df = test_df[test_df["actor_id"].isin(sampled)]

def build_feedback(df, weights):
    df = df.copy()
    df["score"] = df["type"].map(weights).fillna(0) * df["cnt"]
    fb = df.groupby(["actor_id", "repo_id"])["score"].sum().reset_index()
    return fb[fb["score"] > 0]

train_fb = build_feedback(train_df, WEIGHTS)
test_fb = build_feedback(test_df, WEIGHTS)

eval_users_all = sorted(set(train_fb["actor_id"]) & set(test_fb["actor_id"]))
eval_users = list(rng.choice(eval_users_all, size=min(5_000, len(eval_users_all)), replace=False))
test_gt = test_fb.groupby("actor_id")["repo_id"].apply(set).to_dict()

print(f"Train: {len(train_fb):,} interactions ({train_fb['actor_id'].nunique():,} users, {train_fb['repo_id'].nunique():,} items)")
print(f"Test:  {len(test_fb):,} interactions")
print(f"Eval users: {len(eval_users):,}")

In [ ]:
# Index 매핑
user_ids = train_fb["actor_id"].unique()
item_ids = train_fb["repo_id"].unique()
user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {it: i for i, it in enumerate(item_ids)}
idx2item = {i: it for it, i in item2idx.items()}
n_users, n_items = len(user_ids), len(item_ids)

print(f"Users: {n_users:,}, Items: {n_items:,}")

## 2. Item Features

In [ ]:
meta_dict = {}
if DB_PATH.exists():
    conn = sqlite3.connect(str(DB_PATH))
    mdf = pd.read_sql_query(
        "SELECT repo_id, language, stargazers, forks FROM repo_metadata WHERE http_status = 200", conn
    )
    meta_dict = mdf.set_index("repo_id").to_dict(orient="index")
    conn.close()

all_langs = sorted(lang for lang in set(m.get("language") for m in meta_dict.values()) if isinstance(lang, str))
lang2idx = {l: i + 1 for i, l in enumerate(all_langs)}  # 0 = unknown

# (n_items, 3): log_stars, log_forks, lang_idx
item_feat = np.zeros((n_items, 3), dtype=np.float32)
for iid, idx in item2idx.items():
    m = meta_dict.get(iid, {})
    item_feat[idx, 0] = np.log1p(m.get("stargazers", 0) or 0)
    item_feat[idx, 1] = np.log1p(m.get("forks", 0) or 0)
    item_feat[idx, 2] = lang2idx.get(m.get("language"), 0)

has_meta = (item_feat[:, 0] > 0).sum()
print(f"Item features: {item_feat.shape}")
print(f"With metadata: {has_meta:,} / {n_items:,} ({has_meta/n_items:.1%})")
print(f"Languages: {len(lang2idx)}")

class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, n_langs, embed_dim=64):
        super().__init__()
        # User tower
        self.user_embed = nn.Embedding(n_users, embed_dim)
        
        # Item tower
        self.item_embed = nn.Embedding(n_items, embed_dim)
        self.lang_embed = nn.Embedding(n_langs + 1, 8)  # 0 = unknown
        self.item_mlp = nn.Sequential(
            nn.Linear(embed_dim + 8 + 2, embed_dim),  # embed + lang(8) + stars(1) + forks(1)
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        
        nn.init.xavier_uniform_(self.user_embed.weight)
        nn.init.xavier_uniform_(self.item_embed.weight)
    
    def forward(self, user_idx, item_idx, item_feats):
        u = self.user_embed(user_idx)  # (B, 64)
        i_emb = self.item_embed(item_idx)  # (B, 64)
        lang = self.lang_embed(item_feats[:, 2].long())  # (B, 8)
        i_input = torch.cat([i_emb, lang, item_feats[:, :2]], dim=1)  # (B, 74)
        i = self.item_mlp(i_input)  # (B, 64)
        u = nn.functional.normalize(u, dim=1)
        i = nn.functional.normalize(i, dim=1)
        return u, i
    
    def get_item_vectors(self, item_indices, item_feats):
        """전체 아이템 벡터 추출 (추론용)."""
        with torch.no_grad():
            i_emb = self.item_embed(item_indices)
            lang = self.lang_embed(item_feats[:, 2].long())
            i_input = torch.cat([i_emb, lang, item_feats[:, :2]], dim=1)
            i = self.item_mlp(i_input)
            return nn.functional.normalize(i, dim=1)

# CPU 사용 (MPS는 대규모 embedding에서 불안정)
device = torch.device("cpu")
n_langs = len(lang2idx)
model_tt = TwoTower(n_users, n_items, n_langs, embed_dim=64).to(device)

total_params = sum(p.numel() for p in model_tt.parameters())
embed_mem = (n_users + n_items) * 64 * 4 / 1024 / 1024
print(f"Device: {device}")
print(f"Parameters: {total_params:,} (embedding ~{embed_mem:.0f}MB)")
print(model_tt)

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, n_langs, embed_dim=64):
        super().__init__()
        # User tower
        self.user_embed = nn.Embedding(n_users, embed_dim)
        
        # Item tower
        self.item_embed = nn.Embedding(n_items, embed_dim)
        self.lang_embed = nn.Embedding(n_langs + 1, 8)  # 0 = unknown
        self.item_mlp = nn.Sequential(
            nn.Linear(embed_dim + 8 + 2, embed_dim),  # embed + lang(8) + stars(1) + forks(1)
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        
        nn.init.xavier_uniform_(self.user_embed.weight)
        nn.init.xavier_uniform_(self.item_embed.weight)
    
    def forward(self, user_idx, item_idx, item_feats):
        u = self.user_embed(user_idx)  # (B, 64)
        i_emb = self.item_embed(item_idx)  # (B, 64)
        lang = self.lang_embed(item_feats[:, 2].long())  # (B, 8)
        i_input = torch.cat([i_emb, lang, item_feats[:, :2]], dim=1)  # (B, 74)
        i = self.item_mlp(i_input)  # (B, 64)
        u = nn.functional.normalize(u, dim=1)
        i = nn.functional.normalize(i, dim=1)
        return u, i
    
    def get_item_vectors(self, item_indices, item_feats):
        """전체 아이템 벡터 추출 (추론용)."""
        with torch.no_grad():
            i_emb = self.item_embed(item_indices)
            lang = self.lang_embed(item_feats[:, 2].long())
            i_input = torch.cat([i_emb, lang, item_feats[:, :2]], dim=1)
            i = self.item_mlp(i_input)
            return nn.functional.normalize(i, dim=1)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
n_langs = len(lang2idx)
model_tt = TwoTower(n_users, n_items, n_langs, embed_dim=64).to(device)

total_params = sum(p.numel() for p in model_tt.parameters())
print(f"Device: {device}")
print(f"Parameters: {total_params:,}")
print(model_tt)

%%time
# DataLoader
user_t = torch.tensor(train_fb["actor_id"].map(user2idx).values, dtype=torch.long)
item_t = torch.tensor(train_fb["repo_id"].map(item2idx).values, dtype=torch.long)
feat_t = torch.tensor(item_feat[item_t.numpy()], dtype=torch.float32)

BATCH_SIZE = 1024
dataset = TensorDataset(user_t, item_t, feat_t)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

optimizer = torch.optim.Adam(model_tt.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
temperature = 0.05

N_EPOCHS = 5
history = []

print(f"Batches: {len(loader)}, Batch size: {BATCH_SIZE}, Epochs: {N_EPOCHS}")
print(f"Device: {device}")
print()

for epoch in range(N_EPOCHS):
    t0 = time.time()
    model_tt.train()
    total_loss = 0
    for batch_user, batch_item, batch_feat in loader:
        batch_user = batch_user.to(device)
        batch_item = batch_item.to(device)
        batch_feat = batch_feat.to(device)
        
        u_vec, i_vec = model_tt(batch_user, batch_item, batch_feat)
        logits = u_vec @ i_vec.T / temperature
        labels = torch.arange(logits.size(0), device=device)
        loss = nn.functional.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    elapsed = time.time() - t0
    avg_loss = total_loss / len(loader)
    history.append({"epoch": epoch + 1, "loss": avg_loss, "time": elapsed})
    print(f"  Epoch {epoch+1:>2}/{N_EPOCHS}: loss={avg_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6f}, time={elapsed:.1f}s")

print(f"\nTotal training time: {sum(h['time'] for h in history)/60:.1f}min")

In [ ]:
%%time
# DataLoader
user_t = torch.tensor(train_fb["actor_id"].map(user2idx).values, dtype=torch.long)
item_t = torch.tensor(train_fb["repo_id"].map(item2idx).values, dtype=torch.long)
feat_t = torch.tensor(item_feat[item_t.numpy()], dtype=torch.float32)

dataset = TensorDataset(user_t, item_t, feat_t)
loader = DataLoader(dataset, batch_size=4096, shuffle=True, drop_last=True)

optimizer = torch.optim.Adam(model_tt.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
temperature = 0.05

N_EPOCHS = 10
history = []

print(f"Batches: {len(loader)}, Batch size: 4096, Epochs: {N_EPOCHS}")
print()

for epoch in range(N_EPOCHS):
    t0 = time.time()
    model_tt.train()
    total_loss = 0
    for batch_user, batch_item, batch_feat in loader:
        batch_user = batch_user.to(device)
        batch_item = batch_item.to(device)
        batch_feat = batch_feat.to(device)
        
        u_vec, i_vec = model_tt(batch_user, batch_item, batch_feat)
        logits = u_vec @ i_vec.T / temperature
        labels = torch.arange(logits.size(0), device=device)
        loss = nn.functional.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    elapsed = time.time() - t0
    avg_loss = total_loss / len(loader)
    history.append({"epoch": epoch + 1, "loss": avg_loss, "time": elapsed})
    print(f"  Epoch {epoch+1:>2}/{N_EPOCHS}: loss={avg_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6f}, time={elapsed:.1f}s")

print(f"\nTotal training time: {sum(h['time'] for h in history)/60:.1f}min")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([h["epoch"] for h in history], [h["loss"] for h in history], "o-", color="coral")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Two-Tower Training Loss")
plt.tight_layout()
plt.show()

## 5. 임베딩 추출 & FAISS 인덱스

In [ ]:
%%time
import faiss

model_tt.eval()

# User vectors
all_user_idxs = torch.arange(n_users, device=device)
with torch.no_grad():
    user_vectors = model_tt.user_embed(all_user_idxs)
    user_vectors = nn.functional.normalize(user_vectors, dim=1).cpu().numpy()

# Item vectors (배치)
BATCH = 50_000
item_vectors_list = []
all_item_feat = torch.tensor(item_feat, dtype=torch.float32, device=device)
for start in range(0, n_items, BATCH):
    end = min(start + BATCH, n_items)
    idxs = torch.arange(start, end, device=device)
    vecs = model_tt.get_item_vectors(idxs, all_item_feat[start:end])
    item_vectors_list.append(vecs.cpu().numpy())
item_vectors = np.vstack(item_vectors_list).astype(np.float32)

print(f"User vectors: {user_vectors.shape}")
print(f"Item vectors: {item_vectors.shape}")

# Norm 필터 (ALS와 동일하게)
tt_norms = np.linalg.norm(item_vectors, axis=1)
# Two-Tower는 이미 normalize됐으므로 norm ≈ 1.0, 필터 불필요
print(f"Item norm stats: mean={tt_norms.mean():.4f}, std={tt_norms.std():.6f}")

# FAISS 인덱스
tt_index = faiss.IndexFlatIP(64)
tt_index.add(np.ascontiguousarray(item_vectors))
print(f"FAISS index: {tt_index.ntotal:,} items")

## 6. ALS 로드 (비교용)

In [ ]:
%%time
from implicit.als import AlternatingLeastSquares

# ALS 학습 (동일 데이터)
row = train_fb["actor_id"].map(user2idx).values
col = train_fb["repo_id"].map(item2idx).values
data = train_fb["score"].values.astype(np.float32)
train_sparse = sparse.csr_matrix((data, (row, col)), shape=(n_users, n_items))

als_model = AlternatingLeastSquares(factors=64, regularization=0.01, iterations=15, random_state=42)
als_model.fit(train_sparse)
print(f"ALS 학습 완료")

# ALS FAISS 인덱스 (norm 필터)
als_factors = als_model.item_factors
als_norms = np.linalg.norm(als_factors, axis=1)
als_min_norm = max(np.percentile(als_norms[als_norms > 0], 90), 0.1)
als_valid_idxs = np.where(als_norms > als_min_norm)[0]
als_valid = als_factors[als_valid_idxs]
als_normed = (als_valid / np.linalg.norm(als_valid, axis=1, keepdims=True)).astype(np.float32)

als_index = faiss.IndexFlatIP(64)
als_index.add(np.ascontiguousarray(als_normed))
print(f"ALS FAISS index: {als_index.ntotal:,} items (norm > {als_min_norm:.2f})")

## 7. Retrieval 평가: ALS vs Two-Tower

In [ ]:
%%time
import math
from tqdm import tqdm

K_VALUES = [10, 50, 100]
CANDIDATE_K = max(K_VALUES)

train_seen = train_fb.groupby("actor_id")["repo_id"].apply(set).to_dict()
pop_scores = popularity_scores(train_df, WEIGHTS)
pop_candidates = pop_scores.head(CANDIDATE_K + 500).index.tolist()

def precision_recall_ndcg(recommended, relevant, k):
    rec_set = set(recommended[:k])
    hits = rec_set & relevant
    precision = len(hits) / k if k > 0 else 0
    recall = len(hits) / len(relevant) if relevant else 0
    dcg = sum(1.0 / math.log2(i + 2) for i, rid in enumerate(recommended[:k]) if rid in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    ndcg = dcg / idcg if idcg > 0 else 0
    return precision, recall, ndcg

results = {
    name: {k: {"precision": [], "recall": [], "ndcg": []} for k in K_VALUES}
    for name in ["Popularity", "ALS", "Two-Tower"]
}

for uid in tqdm(eval_users, desc="Evaluating"):
    gt = test_gt.get(uid, set())
    if not gt or uid not in user2idx:
        continue
    uidx = user2idx[uid]
    seen = train_seen.get(uid, set())
    
    # Popularity
    pop_recs = [r for r in pop_candidates if r not in seen][:CANDIDATE_K]
    
    # ALS
    als_uvec = als_model.user_factors[uidx].reshape(1, -1)
    als_uvec_n = (als_uvec / (np.linalg.norm(als_uvec) + 1e-9)).astype(np.float32)
    _, als_idxs = als_index.search(als_uvec_n, CANDIDATE_K + 50)
    als_recs = []
    for i in als_idxs[0]:
        if i < 0: continue
        rid = idx2item[als_valid_idxs[i]]
        if rid not in seen:
            als_recs.append(rid)
            if len(als_recs) >= CANDIDATE_K: break
    
    # Two-Tower
    tt_uvec = user_vectors[uidx].reshape(1, -1).astype(np.float32)
    _, tt_idxs = tt_index.search(tt_uvec, CANDIDATE_K + 50)
    tt_recs = []
    for i in tt_idxs[0]:
        if i < 0: continue
        rid = idx2item[i]
        if rid not in seen:
            tt_recs.append(rid)
            if len(tt_recs) >= CANDIDATE_K: break
    
    for model_name, recs in [("Popularity", pop_recs), ("ALS", als_recs), ("Two-Tower", tt_recs)]:
        for k in K_VALUES:
            p, r, n = precision_recall_ndcg(recs, gt, k)
            results[model_name][k]["precision"].append(p)
            results[model_name][k]["recall"].append(r)
            results[model_name][k]["ndcg"].append(n)

print(f"\n{'Model':<14} {'K':>3}  {'Precision':>10} {'Recall':>10} {'NDCG':>10}")
print("-" * 52)
for model_name in ["Popularity", "ALS", "Two-Tower"]:
    for k in K_VALUES:
        p = np.mean(results[model_name][k]["precision"])
        r = np.mean(results[model_name][k]["recall"])
        n = np.mean(results[model_name][k]["ndcg"])
        print(f"{model_name:<14} {k:>3}  {p:>10.5f} {r:>10.5f} {n:>10.5f}")

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {"Popularity": "steelblue", "ALS": "coral", "Two-Tower": "forestgreen"}
bar_width = 0.25

for ax, metric in zip(axes, ["precision", "recall", "ndcg"]):
    x = np.arange(len(K_VALUES))
    for i, model_name in enumerate(["Popularity", "ALS", "Two-Tower"]):
        vals = [np.mean(results[model_name][k][metric]) for k in K_VALUES]
        ax.bar(x + i * bar_width, vals, bar_width, label=model_name, color=colors[model_name])
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels([f"K={k}" for k in K_VALUES])
    ax.set_title(f"{metric.title()}@K")
    ax.legend()

plt.tight_layout()
plt.show()

## 8. 추천 다양성 비교

In [ ]:
# 다양성: 각 모델이 추천한 고유 아이템 수
k = 50
for model_name in ["Popularity", "ALS", "Two-Tower"]:
    unique_items = set()
    for uid in eval_users:
        if uid not in user2idx: continue
        uidx = user2idx[uid]
        seen = train_seen.get(uid, set())
        
        if model_name == "Popularity":
            recs = [r for r in pop_candidates if r not in seen][:k]
        elif model_name == "ALS":
            uvec = als_model.user_factors[uidx].reshape(1, -1)
            uvec_n = (uvec / (np.linalg.norm(uvec) + 1e-9)).astype(np.float32)
            _, idxs = als_index.search(uvec_n, k + 50)
            recs = [idx2item[als_valid_idxs[i]] for i in idxs[0] if i >= 0 and idx2item[als_valid_idxs[i]] not in seen][:k]
        else:
            uvec = user_vectors[uidx].reshape(1, -1).astype(np.float32)
            _, idxs = tt_index.search(uvec, k + 50)
            recs = [idx2item[i] for i in idxs[0] if i >= 0 and idx2item[i] not in seen][:k]
        
        unique_items.update(recs)
    print(f"{model_name:<14} K={k}: {len(unique_items):,} unique items")

## 9. 정성 비교: 유사 Repo

In [ ]:
name_map = pickle.load(open(MODEL_DIR / "repo_name_map.pkl", "rb"))
name_to_id = {v: k for k, v in name_map.items() if isinstance(v, str)}

test_repos = ["pytorch/pytorch", "microsoft/vscode", "openclaw/openclaw"]

for repo_name in test_repos:
    rid = name_to_id.get(repo_name)
    if not rid or rid not in item2idx:
        print(f"{repo_name}: not found")
        continue
    iidx = item2idx[rid]
    
    print(f"\n=== {repo_name} ===")
    
    # ALS
    if iidx < len(als_factors) and als_norms[iidx] > als_min_norm:
        als_v = als_factors[iidx].reshape(1, -1)
        als_v_n = (als_v / np.linalg.norm(als_v)).astype(np.float32)
        scores, idxs = als_index.search(als_v_n, 6)
        print(f"  ALS top-5:")
        for j in range(1, 6):
            if idxs[0][j] < 0: continue
            r = idx2item[als_valid_idxs[idxs[0][j]]]
            print(f"    {name_map.get(r, f'repo_{r}')} ({scores[0][j]:.4f})")
    else:
        print(f"  ALS: norm too low")
    
    # Two-Tower
    tt_v = item_vectors[iidx].reshape(1, -1).astype(np.float32)
    scores, idxs = tt_index.search(tt_v, 6)
    print(f"  Two-Tower top-5:")
    for j in range(1, 6):
        if idxs[0][j] < 0: continue
        r = idx2item[idxs[0][j]]
        print(f"    {name_map.get(r, f'repo_{r}')} ({scores[0][j]:.4f})")

## 10. 모델 저장

In [ ]:
TT_PATH = MODEL_DIR / "two_tower.pt"
torch.save({
    "model_state": model_tt.cpu().state_dict(),
    "n_users": n_users,
    "n_items": n_items,
    "n_langs": n_langs,
    "embed_dim": 64,
    "user2idx": user2idx,
    "item2idx": item2idx,
    "idx2item": idx2item,
    "item_feat": item_feat,
    "history": history,
}, TT_PATH)
print(f"Saved: {TT_PATH} ({TT_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

## 11. 정리

### ALS vs Two-Tower

| | ALS | Two-Tower |
|---|---|---|
| 입력 | interaction only | + language, stars, forks |
| 학습 | CPU, ~1min | MPS GPU, ~23min |
| Norm 필터 | 필요 (92% 아이템 제외) | 불필요 (모든 아이템 정규화) |
| 서빙 | FAISS FlatIP | FAISS FlatIP |

### levit 프로덕션과의 비교

| | levit | 우리 |
|---|---|---|
| User tower | user demo + 방문 임베딩 평균 | user_id embedding |
| Item tower | item embedding (W2V) | item_id + language + stars + forks |
| 학습 | DLRM/Two-Tower + MLflow | PyTorch + MPS |
| 서빙 | Triton | FAISS + Streamlit |